# Building Agents That Use Code
Follows the Hugging Face Agents Course notebook *unit2/smolagents/code_agents*




In [ ]:
!pip install -q -U smolagents "smolagents[litellm]" ddgs markdownify
# [litellm] lets smolagents communicate with Ollama through LiteLLM
# markdownify only needed for VisitWebpageTool in section containing the full Alfred agent. That tool downloads a web page and uses markdownify to convert the HTML to markdown so the agent gets readable text instead of raw HTML. If the package is missing, the tool fails with an error asking us to install it.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.8/36.8 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 9.2 MB/s eta 0:00:00


## Local model server (Ollama)

In [ ]:
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time, requests

server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        requests.get("http://localhost:11434", timeout=1)
        print("Ollama server is up")
        break
    except Exception:
        time.sleep(1)
else:
    print("Server did not start. Re-run this cell.")

Ollama server is up


In [ ]:
# We should ideally use Qwen2.5-Coder-32B but it is too big for a free T4. 14B is the closest size that fits;
# but if it is too slow or runs out of memory, switch to "qwen2.5-coder:7b" (less reliable).
MODEL_TAG = "qwen2.5-coder:14b"
!ollama pull {MODEL_TAG}

## Choose the model backend


In [ ]:
BACKEND = "ollama"   # or "hf"

from smolagents import LiteLLMModel, InferenceClientModel
from smolagents.models import ChatMessage

if BACKEND == "ollama":
    model = LiteLLMModel(
        model_id=f"ollama_chat/{MODEL_TAG}",
        api_base="http://localhost:11434",
        num_ctx=16384,   #agent prompts grow each step; ollama's default context is far too small
    )
else:
    from huggingface_hub import notebook_login
    notebook_login()
    model = InferenceClientModel()

# connection test
print(
    model.generate([
        ChatMessage(
            role="user",
            content=[{"type": "text", "text": "Say hello in one short sentence."}]
        )
    ]).content
)

Hello!


In [ ]:
def show_tokens(agent):
    # Attribute may differ between smolagents versions
    try:
        print("Token usage:", agent.monitor.get_total_token_counts())
    except Exception as e:
        print("Token counter not available:", e)

### Selecting a playlist for the party
- a `CodeAgent` with the DuckDuckGo search tool.
- First tested the tool on its own (no LLM, no cost).

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, tool

search_tool = DuckDuckGoSearchTool(max_results=5)
try:
    print(search_tool("best party music")[:600])
except Exception as e:
    print("DuckDuckGoSearchTool failed:", e)
    # Fallback with the same tool name (web_search), so agent prompts and generated code don't change
    from ddgs import DDGS

    @tool
    def web_search(query: str) -> str:
        """Search the web and return the top results as short text.

        Args:
            query: the search query
        """
        try:
            with DDGS() as d:
                results = d.text(query, max_results=5)
        except Exception as e:
            return f"Search failed: {e}"
        return "\n".join(f"- {r['title']}: {r['body'][:200]} ({r['href']})" for r in results) or "No results found."

    search_tool = web_search

## Search Results

[The 75 Best Party Songs That Will Get Everyone Dancing - Gear4music](https://www.gear4music.com/blog/best-party-songs/)
Creating a good vibe at your event hinges on selecting the perfect soundtrack. So, let's look at the 75 best party songs for your any event.

[Best Songs To Party: DJ's Ultimate Party Songs Playlist](https://top40weekly.com/best-songs-to-party/)
Planning a celebration? Don't forget the music! We've compiled the best songs to party that are sure to get everyone in the mood to dance and have a good time.

[Best Party Songs of All Time - Most Popular Party Da


In [ ]:
agent = CodeAgent(tools=[search_tool], model=model, max_steps=6)
agent.run("Search for the best music recommendations for a party at the Wayne's mansion.")
show_tokens(agent)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the best music recommendations for a party at the Wayne's mansion.                                   │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5-coder:14b ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  party_music_query = "best party music 2023"                                                                      
  party_music_results = web_search(query=party_music_query)                                                        
  print(party_music_results)                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Party Music 2023 Playlist - Best Party Songs 2023 (Today's Best Songs 
...](https://www.youtube.com/playlist?list=PLaWS7LOfGxEBEscRf_TjRm1eNYu_viKkl)
Party Music 2023 Playlist - Best Party Songs 2023 (Today's Best Songs 2023) dance party songs dance songs 2023 
party playlist 2023 best songs to dance to 202...

[Dance Charts 2023 - Top Dance Songs 2023 (EDM & Dance Music 
Playlist)](https://www.youtube.com/playlist?list=PL6LQX_ln70Emw6i8NIGFC_d1HFuYeGglA)
Kickstart your party with these hits. Find our playlist with these keywords: dance charts 2023, dance music 2023, 
top dance songs 2023, dance playlist 2023, ...

[Best of Party-Hits 2023 - playlist by DJ Benni | 
Spotify](https://open.spotify.com/playlist/0xWBnmyPHoyLKZR34q74Xi)
Best of Party-Hits 2023 · Playlist · 215 songs · 713 likes

[30 Best Dance Songs of 2023: Staff Picks - Billboard](https://www.billboard.com/lists/best-dance-songs-2023/)
This year, dance music did what it always does, functioning as a the essential element the ever evolving and often 
joyous scene in which we exist.

[best party songs everyone knows ⭐⭐⭐Party 2026, 2025, 2024, 2023, 2022 
...](https://open.spotify.com/playlist/6G3fFII5kqz800uPlIQazG)
Best Party Hits 2025 - TOP Party Mix 2025 HITS RIGHT NOW!: Pop, EDM & Dance Hits 2025, 2024 & 2023. 2020s + 2010s 
HITS Top Remixes 2023 & 2024 : Viral TIKTOK trends : Your Party Songs 2025 Hits.

Out: None

[Step 1: Duration 7.55 seconds| Input tokens: 2,090 | Output tokens: 85]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Extracting playlist names and URLs                                                                             
  playlists = [                                                                                                    
      {"name": "Party Music 2023 Playlist", "url":                                                                 
  "https://www.youtube.com/playlist?list=PLaWS7LOfGxEBEscRf_TjRm1eNYu_viKkl"},                                     
      {"name": "Dance Charts 2023", "url":                                                                         
  "https://www.youtube.com/playlist?list=PL6LQX_ln70Emw6i8NIGFC_d1HFuYeGglA"},                                     
      {"name": "Best of Party-Hits 2023", "url": "https://open.spotify.com/playlist/0xWBnmyPHoyLKZR34q74Xi"},      
      {"name": "30 Best Dance Songs of 2023", "url": "https://www.billboard.com/lists/best-dance-songs-2023/"},    
      {"name": "best party songs everyone knows ⭐⭐⭐Party 2026, 2025, 2024, 2023, 2022", "url":                  
  "https://open.spotify.com/playlist/6G3fFII5kqz800uPlIQazG"}                                                      
  ]                                                                                                                
                                                                                                                   
  # Creating a formatted string for the final answer                                                               
  recommendations = "\n".join([f"{i+1}. {playlist['name']} - {playlist['url']}" for i, playlist in                 
  enumerate(playlists)])                                                                                           
  final_answer(recommendations)                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 1. Party Music 2023 Playlist - 
https://www.youtube.com/playlist?list=PLaWS7LOfGxEBEscRf_TjRm1eNYu_viKkl
2. Dance Charts 2023 - https://www.youtube.com/playlist?list=PL6LQX_ln70Emw6i8NIGFC_d1HFuYeGglA
3. Best of Party-Hits 2023 - https://open.spotify.com/playlist/0xWBnmyPHoyLKZR34q74Xi
4. 30 Best Dance Songs of 2023 - https://www.billboard.com/lists/best-dance-songs-2023/
5. best party songs everyone knows ⭐⭐⭐Party 2026, 2025, 2024, 2023, 2022 - 
https://open.spotify.com/playlist/6G3fFII5kqz800uPlIQazG

[Step 2: Duration 32.32 seconds| Input tokens: 4,883 | Output tokens: 467]

Token usage: TokenUsage(input_tokens=4883, output_tokens=467, total_tokens=5350)


## Using a custom tool to prepare the menu
The docstring lists the allowed `occasion` values; that keeps the agent from inventing arguments.

In [ ]:
from smolagents import CodeAgent, tool

# Tool to suggest a menu based on the occasion
@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion (str): The type of occasion for the party. Allowed values are:
                        - "casual": Menu for casual party.
                        - "formal": Menu for formal party.
                        - "superhero": Menu for superhero party.
                        - "custom": Custom menu.
    """
    if occasion == "casual":
        return "Pizza, snacks and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

# Alfred, the butler, preparing the menu for the party
agent = CodeAgent(tools=[suggest_menu], model=model, max_steps=5)

# Preparing the menu for the party
agent.run("Prepare a formal menu for the party.")
show_tokens(agent)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Prepare a formal menu for the party.                                                                            │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5-coder:14b ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  menu = suggest_menu(occasion="formal")                                                                           
  final_answer(menu)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 3-course dinner with wine and dessert.

[Step 1: Duration 6.16 seconds| Input tokens: 2,112 | Output tokens: 40]

Token usage: TokenUsage(input_tokens=2112, output_tokens=40, total_tokens=2152)


## Using Python imports inside the agent
Imports outside the safe list are blocked by default. `additional_authorized_imports` allows `datetime`.

In [ ]:
agent = CodeAgent(tools=[], model=model, additional_authorized_imports=["datetime"], max_steps=5)

agent.run(
    """
    Alfred needs to prepare for the party. Here are the tasks:
    1. Prepare the drinks - 30 minutes
    2. Decorate the mansion - 60 minutes
    3. Set up the menu - 45 minutes
    4. Prepare the music and playlist - 45 minutes

    If we start right now, at what time will the party be ready?
    """
)
show_tokens(agent)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Alfred needs to prepare for the party. Here are the tasks:                                                      │
│     1. Prepare the drinks - 30 minutes                                                                          │
│     2. Decorate the mansion - 60 minutes                                                                        │
│     3. Set up the menu - 45 minutes                                                                             │
│     4. Prepare the music and playlist - 45 minutes                                                              │
│                                                                                                                 │
│     If we start right now, at what time will the party be ready?                                                │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5-coder:14b ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import datetime                                                                                                  
                                                                                                                   
  # Current time                                                                                                   
  current_time = datetime.datetime.now()                                                                           
  print(f"Current time: {current_time}")                                                                           
                                                                                                                   
  # Task durations in minutes                                                                                      
  prepare_drinks = 30                                                                                              
  decorate_mansion = 60                                                                                            
  set_up_menu = 45                                                                                                 
  prepare_music = 45                                                                                               
                                                                                                                   
  # Total time required                                                                                            
  total_time = prepare_drinks + decorate_mansion + set_up_menu + prepare_music                                     
  print(f"Total time required: {total_time} minutes")                                                              
                                                                                                                   
  # Calculate the end time                                                                                         
  end_time = current_time + datetime.timedelta(minutes=total_time)                                                 
  final_answer(end_time.strftime("%H:%M"))                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Current time: 2026-10-07 08:20:54.082887
Total time required: 180 minutes

Final answer: 11:20

[Step 1: Duration 12.76 seconds| Input tokens: 2,111 | Output tokens: 178]

Token usage: TokenUsage(input_tokens=2111, output_tokens=178, total_tokens=2289)


## The full Party Preparator agent
Several tools in one agent. `VisitWebpageTool` is capped so a single page can't flood the context.

In [ ]:
from smolagents import (CodeAgent, DuckDuckGoSearchTool, FinalAnswerTool, Tool, tool, VisitWebpageTool)

@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion: The type of occasion for the party.
    """
    if occasion == "casual":
        return "Pizza, snacks and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.

    Args:
        query: A search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    # find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)

    return best_service

class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""

    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }

        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")

try:
    visit_tool = VisitWebpageTool(max_output_length=3000)   # keeps page text (and tokens) small
except TypeError:
    visit_tool = VisitWebpageTool()                          # older smolagents without that argument

In [ ]:
# Alfred, the butler, preparing the party
agent = CodeAgent(
    tools=[
        search_tool,
        visit_tool,
        suggest_menu,
        catering_service_tool,
        SuperheroPartyThemeTool(),
        FinalAnswerTool(),
    ],
    model=model,
    max_steps=10,
    verbosity_level=2,
)

agent.run("Give me the best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme")
show_tokens(agent)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me the best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme    │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5-coder:14b ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: To create the best playlist for a villain masquerade theme party at Wayne's mansion, I will use the       
`superhero_party_theme_generator` tool to get creative ideas for the theme and then use the `catering_service_tool`
to suggest a suitable catering service that aligns with the theme. Finally, I will use the `web_search` tool to    
find playlists that match the villain theme.                                                                       
                                                                                                                   
<code>                                                                                                             
# Generate a villain masquerade theme idea                                                                         
theme_idea = superhero_party_theme_generator(category="villain masquerade")                                        
print(f"Theme Idea: {theme_idea}")                                                                                 
                                                                                                                   
# Suggest a catering service for the villain theme                                                                 
catering_service = catering_service_tool(query="villain-themed catering")                                          
print(f"Catering Service: {catering_service}")                                                                     
                                                                                                                   
# Search for playlists that match the villain theme                                                                
playlist_query = "villain masquerade playlist"                                                                     
playlists = web_search(query=playlist_query)                                                                       
print(f"Playlists: {playlists}")                                                                                   
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Generate a villain masquerade theme idea                                                                       
  theme_idea = superhero_party_theme_generator(category="villain masquerade")                                      
  print(f"Theme Idea: {theme_idea}")                                                                               
                                                                                                                   
  # Suggest a catering service for the villain theme                                                               
  catering_service = catering_service_tool(query="villain-themed catering")                                        
  print(f"Catering Service: {catering_service}")                                                                   
                                                                                                                   
  # Search for playlists that match the villain theme                                                              
  playlist_query = "villain masquerade playlist"                                                                   
  playlists = web_search(query=playlist_query)                                                                     
  print(f"Playlists: {playlists}")                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Theme Idea: Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.
Catering Service: Gotham Catering Co.
Playlists: ## Search Results

[you're at a masquerade and unknowingly dancing with the 
villain](https://open.spotify.com/playlist/4zxYvQs9gByjRhAn5ujSfQ)
you're at a masquerade and unknowingly dancing with the villain · Playlist · 89 songs · 10 likes

[you're at a masquerade and unknowingly dancing with your ...](https://www.youtube.com/watch?v=e1ic7bV9dtA)
you're at a masquerade and unknowingly dancing with your nemesis (dark royalty core playlist)

[Masquerade ball playlist - YouTube 
Music](https://music.youtube.com/playlist?list=PLY6joegkeG1R75YMvOwO6CjVJ1EZQDUJF)
Masquerade ball playlist Erin Antel Playlist • 2026 1.4K views • 39 tracks • 7+ hours Music for any masquerade 
event Sort

[Villain playlist - YouTube Music](https://music.youtube.com/playlist?list=PLbyFIR11RcEnLrxEsLS-IOV096hm7A4yw)
Villain playlist Richard Cleveland Playlist • 2026 135 views • 93 tracks • 4 hours, 57 minutes 😈 Diabolical 
Thoughts

[Vampire the Masquerade thematic playlists : r/vtm - 
Reddit](https://www.reddit.com/r/vtm/comments/vf2aj3/vampire_the_masquerade_thematic_playlists/)
Vampire the Masquerade playlist Dark Mystery music I put together two playlists that focus on dark / eerie vampire 
themes. Hope you enjoy them!

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I have identified the best playlist URL for the villain masquerade theme party at Wayne's mansion. Now, I 
will provide this URL as the final answer.                                                                         
                                                                                                                   
<code>                                                                                                             
final_answer(best_playlist_url)                                                                                    
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(best_playlist_url)                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: https://open.spotify.com/playlist/4zxYvQs9gByjRhAn5ujSfQ

[Step 3: Duration 4.13 seconds| Input tokens: 8,784 | Output tokens: 374]

Token usage: TokenUsage(input_tokens=8784, output_tokens=374, total_tokens=9158)
